In [ ]:
!pip install keras 
!pip install matplotlib
!pip install numpy
!pip install pandas
!pip install tensorflow

In [ ]:
!pip install plotly
!pip install nbformat
%pip install --upgrade "nbformat>=4.2.0"

In [ ]:
import numpy as np
import pandas as pd

import keras

import plotly.express as px

In [ ]:
# loading the data set
flight_dataset = pd.read_csv("C:\\Users\\amitk\\workspace\\Learning_ML\\Linear_Regression\\dataset\\icao_openap_flight_fuel_emissions.csv")

In [ ]:
usable_flight_dataset = flight_dataset.loc[:,["cargo_mass_kg", "oew_kg", "zfw_kg", "tow_kg", "actual_flown_dist_km", "total_fuel_burn_kg"]]

print("Usable component of flight data set")
print(usable_flight_dataset.head(10))

In [ ]:
usable_flight_dataset.info()

In [ ]:
# Training and validation test data split

training_data = usable_flight_dataset.sample(frac=0.95, random_state=42)
eval_data = usable_flight_dataset.drop(training_data.index)

X_Train = training_data.drop(columns=["total_fuel_burn_kg"]).to_numpy()
Y_Train = training_data["total_fuel_burn_kg"].to_numpy()

X_Eval = eval_data.drop(columns=["total_fuel_burn_kg"]).to_numpy()
Y_Eval = eval_data["total_fuel_burn_kg"].to_numpy()

print("Training Data info")
print(training_data.info())

print("\n\n\nEval Data info")
print(eval_data.info())

In [ ]:
training_data.corr()

In [ ]:
px.scatter(training_data, "cargo_mass_kg", "total_fuel_burn_kg")


In [ ]:
px.scatter(training_data, "oew_kg", "total_fuel_burn_kg")


In [ ]:
px.scatter(training_data, "zfw_kg", "total_fuel_burn_kg")


In [ ]:
px.scatter(training_data, "tow_kg", "total_fuel_burn_kg")


In [ ]:
px.scatter(training_data, "actual_flown_dist_km", "total_fuel_burn_kg")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Deifininf a singel neuron model with no activation layer
model = models.Sequential([
    layers.Dense(units=1,input_shape = (X_Train.shape[1],))
])

# Compile the model
model.compile(
    # Optimizer is the strategy to update weights ; The optimizer is the algorithm responsible for adjusting the model's internal weights and biases to reduce error.
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01),

    # Loss is well loss 
    loss='mean_squared_error',

    # Metrics are used to pull out items in a human-readable performance
    metrics=['mean_absolute_error']
)

# View the model structure (it will show exactly 1 layer)
model.summary()

In [ ]:
history = model.fit(
    X_Train, 
    Y_Train, 
    validation_data=(X_Eval, Y_Eval), 
    epochs=50,          # Number of passes through the dataset
    batch_size=256,      # Number of samples per gradient update
    verbose=1           # Progress bar visibility
)


In [ ]:
loss_by_epoch = pd.DataFrame({
    "epoch": range(1, len(history.history["loss"]) + 1),
    "training_loss": history.history["loss"],
    "validation_loss": history.history["val_loss"],
})

print("Mean squared error by epoch:")
print(loss_by_epoch.to_string(index=False))

fig_loss = px.line(
    loss_by_epoch,
    x="epoch",
    y=["training_loss", "validation_loss"],
    log_y=True,
    labels={"value": "Mean squared error (kg^2)", "variable": "Dataset"},
    title="Training and validation loss by epoch",
)
fig_loss.show()

predicted_fuel = model.predict(X_Eval, verbose=0).ravel()
actual_vs_predicted = pd.DataFrame(
    {
        "actual_fuel_burn_kg": Y_Eval,
        "predicted_fuel_burn_kg": predicted_fuel,
    },
    index=eval_data.index,
)

print("Actual vs. predicted fuel burn (first 20 validation rows):")
print(actual_vs_predicted.head(20).to_string())

fig_actual_vs_predicted = px.scatter(
    actual_vs_predicted,
    x="actual_fuel_burn_kg",
    y="predicted_fuel_burn_kg",
    title="Actual vs. predicted fuel burn",
    labels={
        "actual_fuel_burn_kg": "Actual fuel burn (kg)",
        "predicted_fuel_burn_kg": "Predicted fuel burn (kg)",
    },
)
reference_min = actual_vs_predicted.min().min()
reference_max = actual_vs_predicted.max().max()
fig_actual_vs_predicted.add_shape(
    type="line",
    x0=reference_min,
    y0=reference_min,
    x1=reference_max,
    y1=reference_max,
    line={"dash": "dash", "color": "red"},
)
fig_actual_vs_predicted.show()

In [70]:
model.save("my_linear_regression_model.keras")